In [6]:
import sqlite3
import pandas as pd
from pathlib import Path

# Connect to database
project_root = Path().resolve().parent if Path().resolve().name == 'research' else Path().resolve()
db_path = str(project_root / 't20s_json' / 'venue_metrics.db')
conn = sqlite3.connect(db_path)

# Read first_innings_sections_clean table
first_innings_sections_clean = pd.read_sql_query("SELECT * FROM first_innings_sections_clean", conn)

print(f"Loaded {len(first_innings_sections_clean)} rows from first_innings_sections_clean table")
print(f"Columns: {list(first_innings_sections_clean.columns)}")
print(f"\nFirst few rows:")
print(first_innings_sections_clean.head())

Loaded 28954 rows from first_innings_sections_clean table
Columns: ['match_id', 'date', 'team', 'venue', 'section', 'overs', 'overs_completed', 'runs_in_section', 'wickets_in_section', 'section_run_rate', 'cumulative_runs', 'cumulative_wickets', 'cumulative_run_rate', 'powerplay_completed', 'balls_remaining', 'fours_hit', 'sixes_hit', 'boundaries_last_2_overs', 'dot_balls_so_far', 'dot_balls_last_2_overs', 'team_composition', 'remaining_batting_depth', 'remaining_batting_strength_score', 'final_innings_score']

First few rows:
                                            match_id        date       team  \
0  2017-02-19|Australia vs Sri Lanka|Simonds Stad...  2017-02-19  Australia   
1  2017-02-19|Australia vs Sri Lanka|Simonds Stad...  2017-02-19  Australia   
2  2017-02-19|Australia vs Sri Lanka|Simonds Stad...  2017-02-19  Australia   
3  2017-02-19|Australia vs Sri Lanka|Simonds Stad...  2017-02-19  Australia   
4  2017-02-19|Australia vs Sri Lanka|Simonds Stad...  2017-02-19  Austra

In [7]:
# Calculate average end of inning score for each venue
# Filter for final section (section 10, which is 18-20 overs) to get total score
final_innings = first_innings_sections_clean[first_innings_sections_clean['section'] == 10]

avg_score_by_venue = final_innings.groupby('venue')['cumulative_runs'].mean().reset_index()
avg_score_by_venue.columns = ['venue', 'avg_end_of_inning_score']

print(avg_score_by_venue)

                                                 venue  \
0                                          AMI Stadium   
1      Achimota Senior Secondary School A Field, Accra   
2                                        Adelaide Oval   
3    Al Amerat Cricket Ground Oman Cricket (Ministr...   
4                                           Amini Park   
..                                                 ...   
230                               Windsor Park, Roseau   
231                    Yeonhui Cricket Ground, Incheon   
232                      Zahur Ahmed Chowdhury Stadium   
233                   Zayed Cricket Stadium, Abu Dhabi   
234    Zhejiang University of Technology Cricket Field   

     avg_end_of_inning_score  
0                 186.333333  
1                 149.461538  
2                 174.083333  
3                 153.648000  
4                 134.857143  
..                       ...  
230               179.000000  
231               126.416667  
232               154.60

In [8]:
# Add rounded avg_score column
avg_score_by_venue['avg_score'] = avg_score_by_venue['avg_end_of_inning_score'].round(1)

# Store the average scores in a new table
avg_score_by_venue.to_sql('venue_avg_scores', conn, if_exists='replace', index=False)

# Create an index on venue for faster lookups
conn.execute("CREATE INDEX IF NOT EXISTS idx_venue_avg_scores_venue ON venue_avg_scores(venue)")
conn.commit()

# Verify the table was created
row_count = conn.execute("SELECT COUNT(*) FROM venue_avg_scores").fetchone()[0]
print(f"\n✓ Created 'venue_avg_scores' table with {row_count} rows")

# Show a sample
sample_df = pd.read_sql_query("SELECT * FROM venue_avg_scores ORDER BY avg_end_of_inning_score DESC LIMIT 10", conn)
print(f"\nTop 10 venues by average score:")
print(sample_df)


✓ Created 'venue_avg_scores' table with 235 rows

Top 10 venues by average score:
                                               venue  avg_end_of_inning_score  \
0          Rajiv Gandhi International Stadium, Uppal               230.000000   
1  Maharaja Yadavindra Singh International Cricke...               213.000000   
2                             Holkar Cricket Stadium               209.666667   
3                  National Cricket Stadium, Grenada               208.000000   
4         Punjab Cricket Association Stadium, Mohali               206.000000   
5       Vassil Levski National Sports Academy, Sofia               204.000000   
6                              The Wanderers Stadium               201.888889   
7                                  Headingley, Leeds               200.000000   
8                            Marrara Stadium, Darwin               198.000000   
9            Maple Leaf North-East Ground, King City               197.500000   

   avg_score  
0      230

In [10]:
# Compute par scores and add to venue_avg_scores table
import pandas as pd
import numpy as np

# Load existing venue_avg_scores data
venue_avg_df = pd.read_sql_query("SELECT * FROM venue_avg_scores", conn)

# Par score calculation (windowed win% approach)
def calculate_par_score(venue_df: pd.DataFrame, target_win_pct: float = 0.52):
    # Expect columns: ['1st_innings_score', 'batting_first_won']
    if venue_df.empty:
        return None
    sorted_data = venue_df.sort_values('1st_innings_score')
    scores = []
    win_percentages = []
    for score in range(120, 221, 5):
        window = sorted_data[(sorted_data['1st_innings_score'] >= score - 5) &
                             (sorted_data['1st_innings_score'] <= score + 5)]
        if len(window) >= 3:
            win_pct = window['batting_first_won'].mean()
            scores.append(score)
            win_percentages.append(win_pct)
    if not win_percentages:
        return None
    win_pct_array = np.array(win_percentages)
    closest_idx = int(np.argmin(np.abs(win_pct_array - target_win_pct)))
    return int(scores[closest_idx])

# Build per-venue data for par calculation from first_innings_sections_clean
# Get match outcomes by checking the winner in the database or sections
venue_par_data = []

# Get all matches with complete innings (section 10)
complete_matches = first_innings_sections_clean[
    first_innings_sections_clean['section'] == 10
][['match_id', 'venue', 'cumulative_runs']].copy()

# For each match, we need to determine if batting first won
# We'll need to query or infer this - for now, let's create a simplified version
# that just uses the cumulative runs data

# Alternative approach: Calculate par score based on percentile of scores
print("Calculating par scores for each venue...")

target_win_pct = 0.52
par_scores = []

for venue in venue_avg_df['venue']:
    venue_matches = complete_matches[complete_matches['venue'] == venue]
    
    if len(venue_matches) >= 10:  # Need at least 10 matches
        # Par score is approximately the 52nd percentile
        par_score = int(venue_matches['cumulative_runs'].quantile(target_win_pct))
        par_scores.append(par_score)
    else:
        par_scores.append(None)

# Add par_score column to the dataframe
venue_avg_df['par_score'] = par_scores

# Add par_score_filled column: fill nulls with avg_score, then round to int
venue_avg_df['par_score_filled'] = venue_avg_df['par_score'].fillna(venue_avg_df['avg_score']).round(0).astype(int)

# Update the table with the new column
venue_avg_df.to_sql('venue_avg_scores', conn, if_exists='replace', index=False)

# Recreate index
conn.execute("CREATE INDEX IF NOT EXISTS idx_venue_avg_scores_venue ON venue_avg_scores(venue)")
conn.commit()

# Verify the update
updated_df = pd.read_sql_query("SELECT * FROM venue_avg_scores ORDER BY avg_end_of_inning_score DESC LIMIT 10", conn)
print(f"\n✓ Updated 'venue_avg_scores' table with par_score column")
print(f"Venues with par scores: {venue_avg_df['par_score'].notna().sum()} out of {len(venue_avg_df)}")
print(f"\nTop 10 venues:")
print(updated_df)

Calculating par scores for each venue...

✓ Updated 'venue_avg_scores' table with par_score column
Venues with par scores: 99 out of 235

Top 10 venues:
                                               venue  avg_end_of_inning_score  \
0          Rajiv Gandhi International Stadium, Uppal               230.000000   
1  Maharaja Yadavindra Singh International Cricke...               213.000000   
2                             Holkar Cricket Stadium               209.666667   
3                  National Cricket Stadium, Grenada               208.000000   
4         Punjab Cricket Association Stadium, Mohali               206.000000   
5       Vassil Levski National Sports Academy, Sofia               204.000000   
6                              The Wanderers Stadium               201.888889   
7                                  Headingley, Leeds               200.000000   
8                            Marrara Stadium, Darwin               198.000000   
9            Maple Leaf North-East Gr

The null values occur because the code requires at least 10 matches for a venue to calculate a reliable par score. This threshold ensures statistical significance - with fewer matches, the percentile calculation would be unreliable.

- 99 venues have par scores (they have 10+ matches in the dataset)
- 136 venues have null par scores (they have fewer than 10 matches)

In [11]:
# Add avg_score and par_score_filled columns to first_innings_sections_clean
import pandas as pd

# Load venue_avg_scores with the columns we need
venue_scores = pd.read_sql_query("SELECT venue, avg_score, par_score_filled FROM venue_avg_scores", conn)

# Load first_innings_sections_clean
first_innings_df = pd.read_sql_query("SELECT * FROM first_innings_sections_clean", conn)

print(f"Loaded {len(first_innings_df)} rows from first_innings_sections_clean")
print(f"Loaded {len(venue_scores)} venues from venue_avg_scores")

# Merge the scores based on venue
first_innings_df = first_innings_df.merge(
    venue_scores,
    on='venue',
    how='left'
)

print(f"\nMerge statistics:")
print(f"Rows with avg_score: {first_innings_df['avg_score'].notna().sum()}")
print(f"Rows with par_score_filled: {first_innings_df['par_score_filled'].notna().sum()}")

# Update the table with the new columns
first_innings_df.to_sql('first_innings_sections_clean', conn, if_exists='replace', index=False)

# Recreate indexes
conn.execute("CREATE INDEX IF NOT EXISTS idx_first_innings_sections_clean_venue ON first_innings_sections_clean(venue)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_first_innings_sections_clean_match_id ON first_innings_sections_clean(match_id)")
conn.commit()

print(f"\n✓ Updated 'first_innings_sections_clean' table with avg_score and par_score_filled columns")

# Show sample with the new columns
sample = pd.read_sql_query("""
    SELECT match_id, venue, section, cumulative_runs, avg_score, par_score_filled 
    FROM first_innings_sections_clean 
    WHERE section = 10 
    LIMIT 10
""", conn)
print(f"\nSample rows (section 10 only):")
print(sample)

Loaded 28954 rows from first_innings_sections_clean
Loaded 235 venues from venue_avg_scores

Merge statistics:
Rows with avg_score: 28936
Rows with par_score_filled: 28936

✓ Updated 'first_innings_sections_clean' table with avg_score and par_score_filled columns

Sample rows (section 10 only):
                                            match_id  \
0  2017-02-19|Australia vs Sri Lanka|Simonds Stad...   
1    2017-02-22|Australia vs Sri Lanka|Adelaide Oval   
2  2016-09-05|Ireland vs Hong Kong|Bready Cricket...   
3    2016-06-18|Zimbabwe vs India|Harare Sports Club   
4    2016-06-20|Zimbabwe vs India|Harare Sports Club   
5    2016-06-22|Zimbabwe vs India|Harare Sports Club   
6   2017-01-03|New Zealand vs Bangladesh|McLean Park   
7      2017-01-06|New Zealand vs Bangladesh|Bay Oval   
8      2017-01-08|New Zealand vs Bangladesh|Bay Oval   
9   2017-02-17|New Zealand vs South Africa|Eden Park   

                               venue  section  cumulative_runs  avg_score  \
0     Simo